# AIMLC ZG5536 — Large Language Models for Generative AI
#### Assignment 2B — RAG Pipeline (Group 157)
**Domain:** Indian Financial Markets and Digital Payments (RBI / SEBI documents)

This notebook builds a small, complete RAG system on the Assignment 1A corpus: choose a chunk size, compare dense / BM25 / hybrid search, rerank, then generate grounded answers and check them with an LLM judge.

**Working method:** one stage at a time. Each stage is explained, run by the operator, and verified before the next one is added.

## Requirements map (20 marks)

| Part | What is built | Marks | Notebook stage |
|---|---|---|---|
| A | Dense search (FAISS) and chunk-size choice (100 vs 400 tokens) | 5 | Stages 1–3 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 | Stage 4 |
| C | Cross-encoder reranking | 4 | Stage 5 |
| D | Grounded generation + LLM judge | 7 | Stages 6–7 |

**Reused from Assignment 1A (no new data):** the cleaned domain `.txt` files, 10 queries with expected answers from the 1A evaluation split, and the 1A fine-tuned model as the answer generator.

**Requirements authority:** `assignment-2b/concepts/Assignment 2B - RAG Pipeline.pdf`.
**Known PDF inconsistency:** the text says chunk sizes 100 and 400, one table says 100 and 500. This notebook uses **100 and 400** (the text) and discloses the discrepancy in Stage 3.

## How 1A connects to 2B

| Step | What changes | What it does **not** do |
|---|---|---|
| **CPT** (continued pretraining) | All weights keep learning next-token prediction on the finance corpus, so the model's word predictions fit the domain | It does not teach question/answer behaviour |
| **Instruction SFT** | Trains on question→answer pairs so the model answers briefly in the expected style | It does not give the model access to documents at answer time |
| **LoRA / QLoRA** | Trains small low-rank adapter matrices while the base weights stay frozen; QLoRA also stores the frozen base in 4-bit | It is a cheaper way to do SFT, not a different goal |
| **RAG** (this assignment) | At question time, retrieve relevant passages and put them in the prompt so the answer can be grounded in text | It retrains nothing |

A lower perplexity in 1A shows the model predicts domain text better. It does **not** show that the model answers questions correctly or stays faithful to supplied passages. Part D tests that directly.

---
## Stage 0 — Artifact check, environment and configuration

**Goal:** confirm that everything 2B reuses from 1A is present and unchanged, record the compute resources, and fix one configuration that every later stage uses. No retrieval or model loading happens here.

| Cell | Purpose |
|---|---|
| 0A | Configuration: paths, seed, constants; locate this repository's `data/` folder |
| 0B | Resources and package versions (GPU, VRAM, host RAM, disk) |
| 0C | Data integrity: repository `data/` against its SHA-256 manifest, and against the 1A copies on the volume |
| 0D | Model artifacts (CPT model, adapter) checked against the committed hash manifest |
| 0E | Load the corpus and save an input manifest |
| 0F | Model access and disk budget (no downloads yet) |
| 0G | Gate 0 summary |

In [1]:
# Stage 0A: Configuration and path discovery
import os, sys, json, random, hashlib, time, platform
from pathlib import Path

SEED = 42                                   # same seed family as the 1A split (Stage 8E)
random.seed(SEED)

# This repository holds a frozen copy of the 1A text data under data/ (verified by data/MANIFEST.sha256).
# The 1A volume (mounted at /home/group157) still holds the model weights, which are too large for git.
def find_repo_root():
    # Look for data/MANIFEST.sha256 starting from the working directory, then common clone locations.
    # os.getcwd() raises if the kernel's start folder was deleted or re-cloned; restart the kernel in that case.
    starts = []
    try:
        cwd = Path(os.getcwd())
        starts += [cwd, *cwd.parents]
    except FileNotFoundError:
        print("WARNING: the kernel's working directory no longer exists (folder deleted or re-cloned).")
        print("         Restart the kernel from the repository folder. Searching common locations instead.")
    if os.environ.get("REPO_ROOT"):
        starts.append(Path(os.environ["REPO_ROOT"]))
    starts += [Path.home() / "llm-rag", Path("/home/group157/llm-rag")]
    for base in starts:
        if (base / "data" / "MANIFEST.sha256").exists():
            return base
    return None

REPO_ROOT = find_repo_root()
assert REPO_ROOT is not None, "data/MANIFEST.sha256 not found: run this notebook from inside the cloned repository"
DATA_DIR      = REPO_ROOT / "data"
CORPUS_DIR    = DATA_DIR / "corpus"
QA_FILE       = DATA_DIR / "qa" / "instruction_dataset.jsonl"            # as submitted in 1A (Q/A only)
CURATED_FILE  = DATA_DIR / "qa" / "instruction_dataset_curated.jsonl"    # adds source_document and evidence

ARTIFACT_ROOT = Path(os.environ.get("ARTIFACT_ROOT", "/home/group157"))  # 1A volume: weights + original copies
PV_CORPUS_DIR = ARTIFACT_ROOT / "domain_corpus"                          # read-only cross-check, never written
PV_QA_FILE    = ARTIFACT_ROOT / "instruction_dataset.jsonl"              # read-only cross-check, never written
TRAIN_FILE    = DATA_DIR / "qa" / "instruction_dataset_train.jsonl"      # 1A split export (957)
EVAL_FILE     = DATA_DIR / "qa" / "instruction_dataset_eval.jsonl"       # 1A split export (239)
MODELS_DIR    = REPO_ROOT / "models"
MODELS_MANIFEST = MODELS_DIR / "MANIFEST.sha256"                         # hashes of CPT weights and adapter
ADAPTER_DIR   = MODELS_DIR / "qlora_finance_adapter"                     # committed in this repository (about 11 MB)
PV_ADAPTER_DIR = ARTIFACT_ROOT / "qlora_finance_adapter"                 # 1A volume copy, cross-check only
# CPT weights (1.45 GB, not in git). Order: CPT_DIR environment variable, then the read-only backup, then the original.
_cpt_candidates = [os.environ.get("CPT_DIR"),
                   ARTIFACT_ROOT / "model-backups" / "cpt_model_1a",
                   ARTIFACT_ROOT / "cpt_model"]
CPT_DIR = next((Path(c) for c in _cpt_candidates if c and (Path(c) / "model.safetensors").exists()),
               ARTIFACT_ROOT / "cpt_model")
WORK_DIR      = ARTIFACT_ROOT / "assignment_2b_outputs"                  # 2B generated artifacts (not in git)
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DOCS = 30
EXPECTED_QA   = {"total": 1196, "train": 957, "eval": 239}  # from saved 1A outputs (Stage 8E)

# Retrieval constants, fixed by the assignment PDF
CHUNK_SIZES   = [100, 400]       # tokens, no overlap (text of PDF; its table says 500, see intro)
TOP_K_CHUNK_EVAL = 3
TOP_K_COMPARE = 5
TOP_K_FUSE    = 20
RRF_K         = 60
TOP_K_RERANK  = 10
TOP_K_FINAL   = 3

print("Stage 0A: configuration")
print("-" * 40)
print(f"Python         : {platform.python_version()}")
try:
    print(f"Working dir    : {Path(os.getcwd()).resolve()}")
except FileNotFoundError:
    print("Working dir    : (deleted: restart the kernel from the repository folder)")
print(f"Repository root: {REPO_ROOT.resolve()}")
print(f"Data dir       : {DATA_DIR.resolve()}")
print(f"ARTIFACT_ROOT  : {ARTIFACT_ROOT.resolve() if ARTIFACT_ROOT.exists() else ARTIFACT_ROOT}  (exists={ARTIFACT_ROOT.exists()})")
print(f"2B output dir  : {WORK_DIR.resolve()}")

print("\nTop level of ARTIFACT_ROOT (1A volume):")
if ARTIFACT_ROOT.exists():
    for p in sorted(ARTIFACT_ROOT.iterdir()):
        kind = "dir " if p.is_dir() else "file"
        print(f"  {kind}  {p.name}")
else:
    print("  (missing)")

STAGE0 = {}   # check name -> (status, detail), filled by later Stage 0 cells
print("\n>> [OK] Configuration set.")

Stage 0A: configuration
----------------------------------------
Python         : 3.11.11
Working dir    : /home/group157/llm-assignment-2b/llm-rag
Repository root: /home/group157/llm-assignment-2b/llm-rag
Data dir       : /home/group157/llm-assignment-2b/llm-rag/data
ARTIFACT_ROOT  : /home/group157  (exists=True)
2B output dir  : /home/group157/assignment_2b_outputs

Top level of ARTIFACT_ROOT (1A volume):
  dir   .Trash-1000
  dir   .ipynb_checkpoints
  dir   assignment_2b_outputs
  dir   conv-ai
  file  cpt_loss.png
  dir   cpt_model
  dir   cpt_output
  dir   domain_corpus
  file  instruction_dataset.jsonl
  file  instruction_dataset_eval.jsonl
  file  instruction_dataset_train.jsonl
  dir   llm-assignment-1a-finance-corpus
  dir   llm-assignment-2b
  file  llm_baseline_requirements.txt
  dir   model-backups
  dir   qlora_finance_adapter
  dir   qlora_training_output
  file  rag_requirements.txt
  dir   venv
  dir   venv-rag

>> [OK] Configuration set.


In [2]:
# Stage 0B: Resources and package versions
import shutil, importlib, subprocess

print("Stage 0B: compute resources")
print("-" * 40)

# --- GPU ---
gpu_info = {"available": False}
try:
    import torch
    gpu_info["torch"] = torch.__version__
    gpu_info["cuda_build"] = torch.version.cuda
    gpu_info["available"] = torch.cuda.is_available()
    if gpu_info["available"]:
        props = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info(0)
        gpu_info.update(name=props.name, total_gb=round(total_b/1e9, 1), free_gb=round(free_b/1e9, 1),
                        bf16=torch.cuda.is_bf16_supported())
except Exception as e:
    gpu_info["error"] = repr(e)
print("GPU            :", gpu_info)

# --- Host RAM (physical and the container limit, which is what actually binds) ---
def read_first(paths):
    for p in paths:
        try:
            return Path(p).read_text().strip()
        except Exception:
            pass
    return None

mem_total_kb = None
try:
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemTotal"):
            mem_total_kb = int(line.split()[1])
        if line.startswith("MemAvailable"):
            mem_avail_kb = int(line.split()[1])
except Exception:
    mem_avail_kb = None
cg_limit = read_first(["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"])
cg_limit_gb = None
if cg_limit and cg_limit.isdigit() and int(cg_limit) < 1 << 60:
    cg_limit_gb = round(int(cg_limit)/1e9, 1)
print(f"Host RAM       : machine total {round(mem_total_kb/1e6,1) if mem_total_kb else '?'} GB | "
      f"available {round(mem_avail_kb/1e6,1) if mem_avail_kb else '?'} GB | container limit {cg_limit_gb or 'none/unknown'} GB")

# --- Persistent disk ---
disk_target = ARTIFACT_ROOT if ARTIFACT_ROOT.exists() else REPO_ROOT
du = shutil.disk_usage(disk_target)
print(f"Persistent disk: {disk_target}  total {du.total/1e9:.1f} GB | free {du.free/1e9:.1f} GB")
hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
print(f"HF cache dir   : {hf_home}  (exists={hf_home.exists()})")

# --- Package versions ---
PKGS = ["numpy", "torch", "transformers", "tokenizers", "peft", "bitsandbytes", "accelerate",
        "sentence_transformers", "faiss", "rank_bm25", "huggingface_hub", "pandas"]
versions = {}
for name in PKGS:
    try:
        m = importlib.import_module(name)
        versions[name] = getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        versions[name] = None
print("\nPackage versions:")
for k, v in versions.items():
    print(f"  {k:<22}{v if v else 'MISSING'}")

missing = [k for k, v in versions.items() if v is None]
print("\nMissing packages:", missing or "none")
print("If packages are missing: run kubeflow/setup_env.sh in a terminal and select the \"Python (RAG venv)\" kernel.")

(WORK_DIR / "environment_report.json").write_text(json.dumps(
    {"gpu": gpu_info, "mem_total_kb": mem_total_kb, "container_limit_gb": cg_limit_gb,
     "disk_free_gb": round(du.free/1e9, 1), "versions": versions, "python": platform.python_version()}, indent=2))
STAGE0["resources"] = ("PASS" if gpu_info.get("available") else "WARN", f"gpu={gpu_info.get('name','none')}")
print("\n>> [OK] Saved assignment_2b_outputs/environment_report.json")

Stage 0B: compute resources
----------------------------------------
GPU            : {'available': True, 'torch': '2.5.1+cu124', 'cuda_build': '12.4', 'name': 'NVIDIA L40S', 'total_gb': 47.8, 'free_gb': 39.4, 'bf16': True}
Host RAM       : machine total 528.0 GB | available 501.2 GB | container limit 19.3 GB
Persistent disk: /home/group157  total 16002.9 GB | free 5890.5 GB
HF cache dir   : /home/jovyan/.cache/huggingface  (exists=True)

Package versions:
  numpy                 2.4.6
  torch                 2.5.1+cu124
  transformers          4.46.3
  tokenizers            0.20.3
  peft                  0.13.2
  bitsandbytes          0.44.1
  accelerate            1.1.1
  sentence_transformers 3.3.1
  faiss                 1.9.0
  rank_bm25             installed (no __version__)
  huggingface_hub       0.26.2
  pandas                2.2.3

Missing packages: none
If packages are missing: run kubeflow/setup_env.sh in a terminal and select the "Python (RAG venv)" kernel.

>> [OK] Saved 

In [3]:
# Stage 0B-install: optional, run only when a package is missing or a new one is needed.
# Normal route: add the package to kubeflow/requirements.txt and re-run `bash kubeflow/setup_env.sh`
# in a terminal. This cell is the in-notebook shortcut and installs into the SAME venv (the kernel's
# interpreter), so the package persists on the volume.
INSTALL_MISSING = False        # set True, run this cell once, then restart the kernel and re-run from 0A
EXTRA_PACKAGES = []            # e.g. ["rouge-score==0.1.2"]; also add each one to kubeflow/requirements.txt

REQ_PATH = REPO_ROOT / "kubeflow" / "requirements.txt"
if INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_PATH), *EXTRA_PACKAGES]
    print("Running:", " ".join(cmd), "\n(packages that are already satisfied are skipped)\n")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout if r.returncode == 0 else r.stderr)[-1500:])
    print("Done." if r.returncode == 0 else "pip FAILED, see the message above.")
    if EXTRA_PACKAGES:
        print("\nREMINDER: add these to kubeflow/requirements.txt so setup_env.sh and check_env.py include them:", EXTRA_PACKAGES)
    print("Restart the kernel, then re-run from Stage 0A so the new packages are imported.")
else:
    print("INSTALL_MISSING is False: nothing installed.")

INSTALL_MISSING is False: nothing installed.


In [4]:
# Stage 0C: Data integrity (repository data/ vs its manifest, and vs the 1A copies on the volume)
def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def human(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024

def jsonl_rows(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

print("Stage 0C: data integrity")
print("-" * 40)

# 1) Repository data against data/MANIFEST.sha256 (detects any edit to the committed copy)
expected = {}
for line in (DATA_DIR / "MANIFEST.sha256").read_text(encoding="utf-8").splitlines():
    if line.strip():
        digest, rel = line.split("  ", 1)
        expected[rel] = digest
bad, missing_files = [], []
for rel, digest in expected.items():
    p = DATA_DIR / rel
    if not p.exists():
        missing_files.append(rel)
    elif sha256_file(p) != digest:
        bad.append(rel)
on_disk = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()} - {"MANIFEST.sha256", "README.md"}
extra = sorted(on_disk - set(expected))
print(f"Manifest entries: {len(expected)} | missing {len(missing_files)} | changed {len(bad)} | extra {len(extra)}")
for tag, items in [("missing", missing_files), ("CHANGED", bad), ("extra", extra)]:
    for r in items[:10]:
        print(f"  {tag}: {r}")
STAGE0["repo_data_manifest"] = ("PASS" if not (bad or missing_files or extra) else "FAIL",
                                f"{len(expected)} files checked")

corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
STAGE0["corpus"] = ("PASS" if len(corpus_files) == EXPECTED_DOCS else "FAIL", f"{len(corpus_files)} files")

# 2) Q/A files: counts, schema, ordered identity between the two files
qa_rows, cur_rows = jsonl_rows(QA_FILE), jsonl_rows(CURATED_FILE)
same_order = (len(qa_rows) == len(cur_rows) and
              all(a["instruction"] == b["instruction"] and a["response"] == b["response"]
                  for a, b in zip(cur_rows, qa_rows)))
print(f"\nQ/A rows: submitted {len(qa_rows)} | curated {len(cur_rows)} (expected {EXPECTED_QA['total']})")
print("Submitted keys:", sorted(qa_rows[0]), "| curated keys:", sorted(cur_rows[0]))
print("Ordered Q/A identical across the two files:", same_order)
STAGE0["qa_rows"] = ("PASS" if len(qa_rows) == EXPECTED_QA["total"] == len(cur_rows) and same_order else "FAIL",
                     f"{len(qa_rows)} rows")

# 3) Cross-check the 1A volume copies (read-only). A difference means the volume was overwritten or differs.
pv_status = []
if PV_CORPUS_DIR.exists():
    same = diff = absent = 0
    for p in corpus_files:
        q = PV_CORPUS_DIR / p.name
        if not q.exists():
            absent += 1
        elif sha256_file(q) == sha256_file(p):
            same += 1
        else:
            diff += 1
            print("  DIFFERS from volume:", p.name)
    pv_extra = sorted(set(x.name for x in PV_CORPUS_DIR.glob("*.txt")) - set(p.name for p in corpus_files))
    print(f"\nVolume corpus ({PV_CORPUS_DIR}): identical {same} | differs {diff} | absent on volume {absent} | extra on volume {len(pv_extra)}")
    pv_status.append("corpus")
    STAGE0["volume_corpus_matches_repo"] = ("PASS" if diff == 0 and absent == 0 and not pv_extra else "WARN", f"same={same} diff={diff}")
else:
    print(f"\nVolume corpus not present at {PV_CORPUS_DIR} (fine: the repository copy is authoritative).")
    STAGE0["volume_corpus_matches_repo"] = ("PENDING", "no volume copy")
if PV_QA_FILE.exists():
    ok = sha256_file(PV_QA_FILE) == sha256_file(QA_FILE)
    print(f"Volume Q/A file identical to repository copy: {ok}")
    STAGE0["volume_qa_matches_repo"] = ("PASS" if ok else "WARN", "")

# 4) Split exports: counts, disjoint, union equals the full file
train_rows, eval_rows = jsonl_rows(TRAIN_FILE), jsonl_rows(EVAL_FILE)
qkey = lambda r: (r["instruction"], r["response"])
tr_set, ev_set, full_set = set(map(qkey, train_rows)), set(map(qkey, eval_rows)), set(map(qkey, qa_rows))
split_ok = (len(train_rows) == EXPECTED_QA["train"] and len(eval_rows) == EXPECTED_QA["eval"]
            and not (tr_set & ev_set) and (tr_set | ev_set) == full_set)
print(f"\nSplit exports: train {len(train_rows)} (expected {EXPECTED_QA['train']}) | eval {len(eval_rows)} (expected {EXPECTED_QA['eval']})"
      f" | overlap {len(tr_set & ev_set)} | union equals full file: {(tr_set | ev_set) == full_set}")
STAGE0["split_exports"] = ("PASS" if split_ok else "FAIL", f"{len(train_rows)}/{len(eval_rows)}")
print("\n>> [OK] Data integrity checked.")

Stage 0C: data integrity
----------------------------------------
Manifest entries: 34 | missing 0 | changed 0 | extra 0

Q/A rows: submitted 1196 | curated 1196 (expected 1196)
Submitted keys: ['instruction', 'response'] | curated keys: ['evidence', 'instruction', 'response', 'source_document']
Ordered Q/A identical across the two files: True

Volume corpus (/home/group157/domain_corpus): identical 30 | differs 0 | absent on volume 0 | extra on volume 0
Volume Q/A file identical to repository copy: True

Split exports: train 957 (expected 957) | eval 239 (expected 239) | overlap 0 | union equals full file: True

>> [OK] Data integrity checked.


In [5]:
# Stage 0D: Model artifacts, checked against the committed models/MANIFEST.sha256
print("Stage 0D: model artifacts")
print("-" * 40)
model_expected = {}
for line in MODELS_MANIFEST.read_text(encoding="utf-8").splitlines():
    if line.strip():
        digest, rel = line.split("  ", 1)
        model_expected[rel] = digest

def check_dir(label, directory, subdir):
    # Compare every manifest entry under `subdir/` with the files in `directory`.
    if not directory.exists():
        print(f"{label:<28}MISSING: {directory}")
        return "FAIL", "directory missing"
    bad, absent, n = [], [], 0
    entries = [(rel, d) for rel, d in model_expected.items() if rel.startswith(subdir + "/")]
    print(f"{label}: {directory}")
    for k, (rel, digest) in enumerate(entries, start=1):
        n += 1
        p = directory / rel.split("/", 1)[1]
        size = human(p.stat().st_size) if p.exists() else "-"
        print(f"  [{k}/{len(entries)}] checking {rel.split('/', 1)[1]} ({size}) ...", end=" ", flush=True)
        if not p.exists():
            absent.append(rel)
            print("MISSING", flush=True)
        elif sha256_file(p) != digest:
            bad.append(rel)
            print("CHANGED", flush=True)
        else:
            print("OK", flush=True)
    print(f"{'':<2}files {n} | missing {len(absent)} | changed {len(bad)}")
    for r in absent + bad:
        print("   !", r)
    return ("PASS" if not (bad or absent) else "FAIL"), f"{n} files, changed={len(bad)}, missing={len(absent)}"

# Repository adapter (authoritative copy used for generation in Stage 6)
STAGE0["adapter_repo"] = check_dir("Adapter (repository)", ADAPTER_DIR, "qlora_finance_adapter")
ac = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
print("   adapter_config:", {k: ac.get(k) for k in ["base_model_name_or_path", "r", "lora_alpha", "target_modules", "lora_dropout"]})

# CPT model (weights are not in git; this hashes the 1.45 GB weight file, which takes a few seconds)
STAGE0["cpt_model"] = check_dir("CPT model", CPT_DIR, "cpt_model")
cfg_p = CPT_DIR / "config.json"
if cfg_p.exists():
    cfg = json.loads(cfg_p.read_text())
    print("   cpt config:", {k: cfg.get(k) for k in ["architectures", "vocab_size", "max_position_embeddings",
                                                  "hidden_size", "num_hidden_layers", "pad_token_id", "eos_token_id"]})

# Volume adapter copy (cross-check only; optional)
if PV_ADAPTER_DIR.exists():
    STAGE0["adapter_volume"] = check_dir("Adapter (1A volume copy)", PV_ADAPTER_DIR, "qlora_finance_adapter")
else:
    print(f"Adapter (1A volume copy)    not present at {PV_ADAPTER_DIR} (fine: the repository copy is used)")

aw = ADAPTER_DIR / "adapter_model.safetensors"
print(f"\nAdapter weights: {human(aw.stat().st_size)} | tokenizer saved with adapter: {(ADAPTER_DIR / 'tokenizer_config.json').exists()}")
print("\n>> [OK] Model artifacts verified against the committed manifest.")

Stage 0D: model artifacts
----------------------------------------
Adapter (repository): /home/group157/llm-assignment-2b/llm-rag/models/qlora_finance_adapter
OK[1/8] checking README.md (5.0 KB) ... 
  [2/8] checking adapter_config.json (635.0 B) ... OK
  [3/8] checking adapter_model.safetensors (6.3 MB) ... OK
  [4/8] checking merges.txt (455.5 KB) ... OK
  [5/8] checking special_tokens_map.json (864.0 B) ... OK
  [6/8] checking tokenizer.json (3.4 MB) ... OK
  [7/8] checking tokenizer_config.json (3.9 KB) ... OK
  [8/8] checking vocab.json (781.9 KB) ... OK
  files 8 | missing 0 | changed 0
   adapter_config: {'base_model_name_or_path': 'cpt_model', 'r': 16, 'lora_alpha': 32, 'target_modules': ['v_proj', 'q_proj'], 'lora_dropout': 0.05}
CPT model: /home/group157/model-backups/cpt_model_1a
OK[1/8] checking config.json (777.0 B) ... 
  [2/8] checking generation_config.json (111.0 B) ... OK
  [3/8] checking merges.txt (455.5 KB) ... OK
  [4/8] checking model.safetensors (1.3 GB) ... OK


In [6]:
# Stage 0E: Load the corpus and save an input manifest
docs = {}
manifest = []
t0 = time.perf_counter()
print(f"Loading {len(corpus_files)} documents from {CORPUS_DIR} ...\n")
for i, p in enumerate(corpus_files, start=1):
    print(f"[{i:>2}/{len(corpus_files)}] loading {p.name[:80]} ...", end=" ", flush=True)
    text = p.read_text(encoding="utf-8")
    docs[p.name] = text
    manifest.append({"doc_id": p.name, "chars": len(text), "bytes": p.stat().st_size, "sha256": sha256_file(p)})
    print(f"OK ({len(text):,} chars)", flush=True)
print(f"\nAll {len(docs)} documents loaded in {time.perf_counter() - t0:.1f} s\n")

assert len(docs) == EXPECTED_DOCS, f"expected {EXPECTED_DOCS} documents, found {len(docs)}"

total_chars = sum(m["chars"] for m in manifest)
print("Stage 0E: corpus loaded")
print("-" * 40)
print(f"Documents      : {len(docs)}")
print(f"Total characters: {total_chars:,}")
print("\nFive largest / five smallest documents (characters):")
by_size = sorted(manifest, key=lambda m: m["chars"])
for m in by_size[:5] + by_size[-5:]:
    print(f"  {m['chars']:>10,}  {m['doc_id'][:90]}")

provenance = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "artifact_root": str(ARTIFACT_ROOT), "seed": SEED, "documents": manifest,
    "qa_file_sha256": sha256_file(QA_FILE), "curated_file_sha256": sha256_file(CURATED_FILE),
}
(WORK_DIR / "input_manifest.json").write_text(json.dumps(provenance, indent=2))
STAGE0["docs_loaded"] = ("PASS", f"{len(docs)} docs, {total_chars:,} chars")
print("\n>> [OK] Saved assignment_2b_outputs/input_manifest.json")

Loading 30 documents from /home/group157/llm-assignment-2b/llm-rag/data/corpus ...

OK (138,593 chars)ofitability of Individual Traders in the Equity Derivatives Segment (FY25–FY26 ... 
[ 2/30] loading RBI - Annual Report 2023-24.txt ... OK (805,274 chars)
[ 3/30] loading RBI - Annual Report 2025-26.txt ... OK (612,611 chars)
[ 4/30] loading RBI - Asset Reconstruction Companies Directions 2025.txt ... OK (130,075 chars)
[ 5/30] loading RBI - Asset Reconstruction Companies – Credit Information Reporting Directions 2 ... OK (74,228 chars)
[ 6/30] loading RBI - Asset Reconstruction Companies – Know Your Customer Directions 2025.txt ... OK (163,517 chars)
[ 7/30] loading RBI - Asset Reconstruction Companies – Treatment of Wilful Defaulters and Large  ... OK (17,248 chars)
[ 8/30] loading RBI - Benchmarking Indias Payments Systems - 2022.txt ... OK (138,602 chars)
[ 9/30] loading RBI - Digital Lending Guidelines.txt ... OK (20,132 chars)
[10/30] loading RBI - Digitalisation and Payment Revo

In [7]:
# Stage 0F: Model access and disk budget (no weights are downloaded here)
PLANNED_MODELS = [
    # (role, model id, stage, intended precision)
    ("embedder",            "BAAI/bge-small-en-v1.5",                  "Stage 2",  "fp32/fp16 on GPU"),
    ("cross-encoder",       "cross-encoder/ms-marco-MiniLM-L-6-v2",    "Stage 5",  "fp32/fp16 on GPU"),
    ("cross-encoder (alt)", "cross-encoder/ms-marco-MiniLM-L6-v2",     "Stage 5",  "fp32/fp16 on GPU"),
    ("generator fallback",  "Qwen/Qwen2.5-7B-Instruct",                "Stage 6",  "bf16 or 4-bit"),
    ("judge candidate",     "meta-llama/Llama-3.1-8B-Instruct",        "Stage 7",  "bf16 or 4-bit"),
]
print("Stage 0F: model access and download size (metadata only)")
print("-" * 60)
rows = []
try:
    from huggingface_hub import HfApi
    api = HfApi()
    for role, mid, stage, prec in PLANNED_MODELS:
        try:
            info = api.model_info(mid, files_metadata=True)
            weights = [s for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin")) and s.size]
            size_gb = sum(s.size for s in weights) / 1e9
            gated = bool(getattr(info, "gated", False))
            rows.append((role, mid, stage, prec, "reachable", f"{size_gb:.2f} GB", "gated" if gated else "open"))
        except Exception as e:
            rows.append((role, mid, stage, prec, "NOT reachable", "?", type(e).__name__))
except Exception as e:
    print("huggingface_hub unavailable or network blocked:", repr(e))

for r in rows:
    print(f"{r[0]:<20}{r[1]:<40}{r[2]:<8}{r[3]:<18}{r[4]:<15}{r[5]:<10}{r[6]}")

free_gb = shutil.disk_usage(hf_home if hf_home.exists() else Path.home()).free / 1e9
print(f"\nFree disk where the HF cache lives: {free_gb:.1f} GB")
print("Reminder: a 4-bit runtime load still downloads the full-precision checkpoint unless a pre-quantized repo is used.")
print("Gated models (e.g. Llama) need an accepted licence and an HF token. Resolve this before Stage 7.")

ok = bool(rows) and any(r[4] == "reachable" for r in rows)
STAGE0["model_access"] = ("PASS" if ok else "WARN", f"{sum(r[4]=='reachable' for r in rows)}/{len(rows)} reachable")
print("\n>> [OK] Access check complete. Nothing downloaded.")

Stage 0F: model access and download size (metadata only)
------------------------------------------------------------
embedder            BAAI/bge-small-en-v1.5                  Stage 2 fp32/fp16 on GPU  reachable      0.27 GB   open
cross-encoder       cross-encoder/ms-marco-MiniLM-L-6-v2    Stage 5 fp32/fp16 on GPU  reachable      0.30 GB   open
cross-encoder (alt) cross-encoder/ms-marco-MiniLM-L6-v2     Stage 5 fp32/fp16 on GPU  reachable      0.30 GB   open
generator fallback  Qwen/Qwen2.5-7B-Instruct                Stage 6 bf16 or 4-bit     reachable      15.23 GB  open
judge candidate     meta-llama/Llama-3.1-8B-Instruct        Stage 7 bf16 or 4-bit     reachable      16.06 GB  gated

Free disk where the HF cache lives: 2485.8 GB
Reminder: a 4-bit runtime load still downloads the full-precision checkpoint unless a pre-quantized repo is used.
Gated models (e.g. Llama) need an accepted licence and an HF token. Resolve this before Stage 7.

>> [OK] Access check complete. Nothing dow

In [8]:
# Stage 0G: Gate 0 summary
print("Stage 0 gate summary")
print("-" * 60)
for k, (status, detail) in STAGE0.items():
    print(f"{status:<8}{k:<30}{detail}")

hard_fail = [k for k, (s, _) in STAGE0.items() if s == "FAIL"]
print()
if hard_fail:
    print("Gate 0 NOT passed. Resolve:", hard_fail)
else:
    print("No hard failures. Review any WARN / PENDING rows with the reviewer before starting Stage 1.")

_ = (WORK_DIR / "stage0_summary.json").write_text(json.dumps(STAGE0, indent=2))   # "_ =" stops Jupyter echoing the byte count
print(f"Saved {WORK_DIR.name}/stage0_summary.json")

Stage 0 gate summary
------------------------------------------------------------
PASS    resources                     gpu=NVIDIA L40S
PASS    repo_data_manifest            34 files checked
PASS    corpus                        30 files
PASS    qa_rows                       1196 rows
PASS    volume_corpus_matches_repo    same=30 diff=0
PASS    volume_qa_matches_repo        
PASS    split_exports                 957/239
PASS    adapter_repo                  8 files, changed=0, missing=0
PASS    cpt_model                     8 files, changed=0, missing=0
PASS    adapter_volume                8 files, changed=0, missing=0
PASS    docs_loaded                   30 docs, 7,160,100 chars
PASS    model_access                  5/5 reachable

No hard failures. Review any WARN / PENDING rows with the reviewer before starting Stage 1.
Saved assignment_2b_outputs/stage0_summary.json
